**This part is required for all the notebook**

In [1]:
import sys
sys.path.append("lib")

from main import Notebook, HandsOn2CatsnifferUI

nb = Notebook()

# Notes:
# Fix typing error:
# pip install --upgrade typing_extensions
# !pip install 

# NOTE:
This part of the notebook requires internet connection, please run only once.

# Requirements
## Clone or update Catsniffer-Tools
This code cell will execute part of the CatSniffer setup process. It checks out or updates the CatSniffer repository containing firmware and utilities.

> **Please run the following code block before begin the Labs!**

In [ ]:
nb.clone_catsniffer_tools()

## Installing the Requierements
This code cell will execute part of the CatSniffer Catnip tool to download the last firmware from the repo.

In [ ]:
nb.install_catsniffer_requirements()

## Download Firmware files
This code cell will execute part of the CatSniffer Catnip tool process. It checks out and download the CatSniffer firmware.

In [ ]:
nb.download_catsniffer_firmware()

---

# Hands-on Lab #2: Sniffing Distributed ZigBee Networks

## Objectives
- Learn the basics of ZigBee network topology (coordinator, routers, end devices)
- Use CatSniffer to detect and follow ZigBee mesh routing patterns
- Identify cluster and endpoint addresses in ZigBee payloads

## Challenge
Capture packets from a distributed ZigBee network. Map the network based on MAC addresses and routing behavior. Determine the role of each node and identify the clusters being used.

## Walk-Through
First of all we need to flash our RP2040 to control the CC1352P7 with the firmware `SerialPassthroughwithboot_RP2040_v1.1.uf2`

![Catsniffer Components Diagram](static/block_diagrams.png)

### Flash firmware

> Run the next code, and after that follow the steps to put the bootloader mode


In [ ]:
serialpass_file = "serialpassthroughwithboot_rp2040_v1.1.uf2"
nb.flash_uf2_firmware(serialpass_file)


### RP2040 bootloader mode:

![Catsniffer Board](static/banner.jpg)


- Press and hold boot RP2040
- While Holding Boot RP2040, press Reset RP2040
- The device will be shown as a thumbdrive

![RP 2040 Boot](static/RP_USB.png)


### Flashing CC1352:

For this lab, we will use the `sniffer_fw_cc1352p_7_v1.10.hex`, this is a Multiprotocol sniffer from Texas Instrument.

In [ ]:
sniffer_file = "sniffer_fw_cc1352p_7_v1.10.hex"
nb.flash_cc_fiwmare(sniffer_file)

**Note**: A *successful* upload will produce an output similar to:
```bash
Firmware uploaded successfully.

[SUCCESS] Firmware loaded successfully.
```

If you see anything different, please contact the trainer.

### Running the Sniffer
1. Start sniffing with the CatSniffer and the PyCatSniffer tool using the following command:

```bash
python cat_sniffer.py sniff <SERIAL_PORT> -ff -ws -phy zigbee -c 25
```

In [2]:
lab2ui = HandsOn2CatsnifferUI()
lab2ui.display_ui_terminal()

2. Once Wireshark is running, we can view the encrypted communication between the end devices and the Zigbee Coordinator The first step to finding a vulnerable target is to use the Zigbee key from the Zigbee Alliance: 
- Go to Edit > Preference > Protocols
- Zigbee > Pre-configured Keys > Edit…
- Add the following new record:
  - Key: `5A:69:67:42:65:65:41:6C:6C:69:61:6E:63:65:30:39`
  - Byte Order: Normal
  - Label: Optional
- Close the modal dialog with the “OK” Button

![Wireshark Configuration Zigbee key](static/ws_config.png)

3. When we close the dialog, we can see the first decryption layer of the Zigbee devices. This layer sees the interaction as “Commands” in the Info column. 

![Wireshark Sniffer TI Running](static/ws_sniffing.png)

When we are sniffing a distributed network, we need to wait until a device tries to reconnect or a new device wants to join the network. When the Zigbee coordinator initiates the pairing process, it sends announcement packets. The end device reads these packets, and the pairing process starts. Wait until you identify a pairing process. You should see **Transport Key** in the information column:


![Wireshark Transport Key](static/ws_transport.png)

4. Use the key obtained in the “*Transport Key*” message and add it similarly to **Step 2**. With this new key, we have complete decryption of the communication between the end device and the coordinator, and we can view the Application/Device Profile and the cluster information


![Wireshark Full Decrypted Communication](static/ws_analyse.png)

Analyze the Zigbee packets again to see the fully decrypted messages and observe the structure of some of the commands.